In [9]:
import pandas as pd
import numpy as np
from sqlalchemy import MetaData, Table, Column, Integer, String, Date, Numeric, text,create_engine

In [10]:
db_connection_string = (r'mssql+pyodbc://DESKTOP-LEVIKVU\SQLEXPRESS/E-Commerce_Sales_Analytics?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes')
engine = create_engine(db_connection_string,fast_executemany=True)
conn = engine.connect()

In [11]:
# Ask SQL to do the heavy lifting before it reaches Python
query = """
    SELECT * FROM historical_sales_data 
    WHERE YEAR(order_date) = (SELECT MAX(YEAR(order_date)) FROM historical_sales_data)
    AND MONTH(order_date) = (SELECT MAX(MONTH(order_date)) FROM historical_sales_data)
"""
monthly_summary = pd.read_sql(query, engine)

In [19]:
monthly_summary.head()

0          0.0
1          0.0
2          0.0
3          0.0
4          0.0
         ...  
3463       0.0
3464    SAVE46
3465       0.0
3466    SAVE30
3467       0.0
Name: coupon_code, Length: 3468, dtype: object

In [6]:
#df = pd.read_sql_table("historical_sales_data",engine)

In [7]:
#max_date = df['order_date'].max()

In [8]:
#max_date

In [9]:
#monthly_summary = df[(df['order_date'].dt.year == max_date.year) & (df['order_date'].dt.month == max_date.month)]

In [10]:
monthly_summary = monthly_summary.sort_values('order_date', ascending=True)

In [11]:
monthly_summary.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-898159,2025-12-01,11:19:47,Cancelled,Mobile App,CUST-023404,Michael Leach,71,Female,VIP,...,0.00,121.49,23.76,1880.82,797.46,1059.60,56.34,10963.77,True,5
151,ORD-600401,2025-12-01,02:02:05,Completed,Mobile App,CUST-004601,John Joseph,40,Male,Consumer,...,226.33,35.08,17.88,554.19,283.34,252.97,45.65,10247.42,True,6
152,ORD-792860,2025-12-01,12:32:30,Returned,Mobile App,CUST-013103,Pamela Morgan,63,Male,Consumer,...,0.00,32.07,9.05,499.37,276.91,213.41,42.74,3928.52,True,4
153,ORD-997639,2025-12-01,16:39:05,Completed,Website,CUST-023159,Jennifer Lowe,30,Female,Consumer,...,75.35,45.17,24.75,295.76,129.35,141.66,47.90,5704.66,True,6
154,ORD-314999,2025-12-01,22:25:24,Completed,Mobile App,CUST-015827,Kenneth Williams,57,Female,Consumer,...,157.34,210.07,5.20,1265.60,830.76,429.64,33.95,12478.27,True,10


In [12]:
summary_tbl = monthly_summary.groupby(["order_date","sales_channel","marketing_channel"], as_index=False).agg(
    {'net_sales':'sum'}
)

In [13]:
summary_tbl.head()

,order_date,sales_channel,marketing_channel,net_sales
0,2025-12-01,Marketplace,Affiliate,2822.45
1,2025-12-01,Marketplace,Direct,3387.63
2,2025-12-01,Marketplace,Facebook Ads,2344.96
3,2025-12-01,Marketplace,Instagram,3620.71
4,2025-12-01,Marketplace,Organic Search,4073.46


#### Develpment of schema

In [15]:
metadata = MetaData()

summary_tbl_schema = Table(

    'latest_month_daily_sales',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True), # Enforced Primary Key
    Column('order_date', Date, nullable=False),                  # Enforced Date type
    Column('sales_channel', String(50)),                         # Enforced character limit
    Column('marketing_channel', String(50)), 
    Column('net_sales', Numeric(10, 2))                          # Enforced financial precision
)



In [16]:
#Drop the existing table and recreate it with the strict rules above
# summary_tbl_schema.drop(engine, checkfirst=True)


#metadata.create_all(engine) is the specific command that physically builds the table structure in your SQL Server database.
# metadata.create_all(engine)

In [17]:
summary_tbl.head()

,order_date,sales_channel,marketing_channel,net_sales
0,2025-12-01,Marketplace,Affiliate,2822.45
1,2025-12-01,Marketplace,Direct,3387.63
2,2025-12-01,Marketplace,Facebook Ads,2344.96
3,2025-12-01,Marketplace,Instagram,3620.71
4,2025-12-01,Marketplace,Organic Search,4073.46


In [18]:
# Why use begin() instead of connect()?
# If you used with engine.connect() as conn:, it would open the connection,
# but it would not automatically commit your transaction. 
# You would have to manually write conn.commit() at the end of the block. Using engine.begin() 
# is the industry standard shortcut for database modifications because it handles the commit 
# and the error-handling rollback for you automatically.

In [43]:
with engine.begin() as conn :
    conn.execute(text("TRUNCATE TABLE latest_month_daily_sales"))

#### push data in latest_month_daily_sales table in sql

In [21]:
summary_tbl.to_sql(
    name='latest_month_daily_sales',
    con=engine,
    if_exists='append',
    index=False
)

-1